# 01 - Exploratory Data Analysis

读取 `data/raw_data/train.jsonl`（11.3 GB）的思路：

1. **小样本探索**（秒级）：只读前 N 个会话，验证格式、跑通流程。
2. **全量单遍统计**（分钟级，内存小）：流式扫描整文件，只做增量统计，不展开全部事件。
3. **jsonl → parquet**（推荐）：一次性分片转换，之后 EDA 全部基于 parquet，快一个量级。

> 首次建议先执行「冒烟测试」版本（`limit=100_000`）验证，再决定是否全量跑。

In [1]:
import json
from collections import Counter
from pathlib import Path

import os
import pandas as pd
from tqdm.auto import tqdm

os.chdir(r"E:\Algorithm\project\OTTO")  # make data/... resolve against the OTTO project root
DATA_FILE = Path("data/raw_data/train.jsonl")

d:\Anaconda\envs\recbole310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 小样本探索（秒级）

In [2]:
def load_head(n_sessions: int = 100_000) -> pd.DataFrame:
    """读取前 n_sessions 个会话，展平为长表 (session, aid, ts, type, locale)。"""
    rows = []
    with open(DATA_FILE, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i >= n_sessions:
                break
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            for ev in rec["events"]:
                rows.append((rec["session"], ev["aid"], ev["ts"], ev["type"], rec.get("locale")))
    df = pd.DataFrame(rows, columns=["session", "aid", "ts", "type", "locale"])
    df["session"] = df["session"].astype("int64")
    df["aid"] = df["aid"].astype("int32")
    df["ts"] = df["ts"].astype("int64")
    df["type"] = df["type"].astype("category")
    return df


df = load_head(100_000)
df.head()

,session,aid,ts,type,locale
0,0,1517085,1659304800025,clicks,None
1,0,1563459,1659304904511,clicks,None
2,0,1309446,1659367439426,clicks,None
3,0,16246,1659367719997,clicks,None
4,0,1781822,1659367871344,clicks,None


In [3]:
print("事件行数:", len(df))
print("会话数:", df["session"].nunique())
print("商品数:", df["aid"].nunique())
print()
print("type 计数（事件数，非会话数）:")
print(df["type"].value_counts())
print()
print("locale 计数（含缺失）:")
print(df["locale"].value_counts(dropna=False))

事件行数: 5227653
会话数: 100000
商品数: 663079

type 计数（事件数，非会话数）:
type
clicks    4770172
carts      364579
orders      92902
Name: count, dtype: int64

locale 计数（含缺失）:
locale
None    5227653
Name: count, dtype: int64


In [4]:
# 会话长度分布（注意 OTTO 有超长会话，看分位数而不是只看均值）
len_dist = df.groupby("session").size()
print(len_dist.describe(percentiles=[0.5, 0.9, 0.99]))

# 时间跨度
print()
print("ts 范围:", df["ts"].min(), "->", df["ts"].max())
print("跨度(天):", (df["ts"].max() - df["ts"].min()) / 86_400)

# 抽查一个会话的序列结构
sid = df["session"].iloc[0]
df[df["session"] == sid].sort_values("ts").head(10)

count    100000.00000
mean         52.27653
std          76.86172
min           2.00000
50%          19.00000
90%         152.10000
99%         362.00000
max         495.00000
dtype: float64

ts 范围: 1659304800025 -> 1661723999941
跨度(天): 27999.999027777776


,session,aid,ts,type,locale
0,0,1517085,1659304800025,clicks,None
1,0,1563459,1659304904511,clicks,None
2,0,1309446,1659367439426,clicks,None
3,0,16246,1659367719997,clicks,None
4,0,1781822,1659367871344,clicks,None
5,0,1152674,1659367885796,clicks,None
6,0,1649869,1659369893840,carts,None
7,0,461689,1659369898050,carts,None
8,0,305831,1659370027105,orders,None
9,0,461689,1659370027105,orders,None


## 2. 全量单遍统计（分钟级，内存小）

只做增量统计（Counter / set / min-max），不展开所有事件。
**11 GB 需要几分钟，可以先不跑这个单元格，直接跳到第 3 步转 parquet。**

In [ ]:
def full_scan_stats() -> dict:
    """单遍扫描整个 jsonl，返回增量统计结果。"""
    stats = {
        "n_sessions": 0,
        "n_events": 0,
        "types": Counter(),
        "locales": Counter(),
        "ts_min": None,
        "ts_max": None,
    }
    items = set()  # 185 万 aid 约几十 MB，可接受
    with open(DATA_FILE, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="scanning jsonl", unit="line"):
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            stats["n_sessions"] += 1
            for ev in rec["events"]:
                stats["n_events"] += 1
                stats["types"][ev["type"]] += 1
                items.add(ev["aid"])
                if stats["ts_min"] is None or ev["ts"] < stats["ts_min"]:
                    stats["ts_min"] = ev["ts"]
                if stats["ts_max"] is None or ev["ts"] > stats["ts_max"]:
                    stats["ts_max"] = ev["ts"]
            stats["locales"][rec.get("locale")] += 1
    stats["n_items"] = len(items)
    return stats


# 全量统计（分钟级；确认要跑再取消注释）
# full_stats = full_scan_stats()
# full_stats

## 3. jsonl → parquet（推荐，之后 EDA 全用 parquet）

In [5]:
def jsonl_to_parquet(input_path, output_dir, shard_sessions: int = 500_000, limit: int | None = None):
    """流式展平 jsonl 并分片写 parquet，避免一次性加载全量。

    返回 (会话数, 事件总数, 输出文件列表)。
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    rows, files, n_sessions, n_total = [], [], 0, 0

    def flush(idx: int) -> None:
        df = pd.DataFrame(rows, columns=["session", "aid", "ts", "type", "locale"])
        df["session"] = df["session"].astype("int64")
        df["aid"] = df["aid"].astype("int32")
        df["ts"] = df["ts"].astype("int64")
        df["type"] = df["type"].astype("category")
        path = output_dir / f"events_{idx:04d}.parquet"
        df.to_parquet(path, index=False)
        files.append(path)

    with open(input_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="jsonl -> parquet", unit="line"):
            if limit is not None and n_sessions >= limit:
                break
            line = line.strip()
            if not line:
                continue
            rec = json.loads(line)
            for ev in rec["events"]:
                rows.append((rec["session"], ev["aid"], ev["ts"], ev["type"], rec.get("locale")))
                n_total += 1
            n_sessions += 1
            if n_sessions % shard_sessions == 0:
                flush(len(files))
                rows.clear()
    if rows:
        flush(len(files))
    return n_sessions, n_total, files

In [8]:
# 冒烟测试：先转 10 万会话验证流程（秒级~分钟级）
n_sessions, n_total, files = jsonl_to_parquet(
    DATA_FILE, Path("data/processed/events"), shard_sessions=200_000, limit=None
)
print(f"{n_sessions} sessions, {n_total} events -> {len(files)} shard(s)")
files

jsonl -> parquet: 12899779line [05:07, 41964.45line/s] 


12899779 sessions, 216716096 events -> 65 shard(s)


[WindowsPath('data/processed/events/events_0000.parquet'),
 WindowsPath('data/processed/events/events_0001.parquet'),
 WindowsPath('data/processed/events/events_0002.parquet'),
 WindowsPath('data/processed/events/events_0003.parquet'),
 WindowsPath('data/processed/events/events_0004.parquet'),
 WindowsPath('data/processed/events/events_0005.parquet'),
 WindowsPath('data/processed/events/events_0006.parquet'),
 WindowsPath('data/processed/events/events_0007.parquet'),
 WindowsPath('data/processed/events/events_0008.parquet'),
 WindowsPath('data/processed/events/events_0009.parquet'),
 WindowsPath('data/processed/events/events_0010.parquet'),
 WindowsPath('data/processed/events/events_0011.parquet'),
 WindowsPath('data/processed/events/events_0012.parquet'),
 WindowsPath('data/processed/events/events_0013.parquet'),
 WindowsPath('data/processed/events/events_0014.parquet'),
 WindowsPath('data/processed/events/events_0015.parquet'),
 WindowsPath('data/processed/events/events_0016.parquet'

## 4. 之后从 parquet 读取（比扫 jsonl 快一个量级）

全量转换：把 `limit` 去掉、`output_dir` 改为 `data/processed/events` 即可（11 GB，耗时较长，建议在脚本/后台跑）。

In [7]:
# 读取单片 parquet（后续 EDA 全部基于 parquet）
events = pd.read_parquet(Path("data/processed/smoke") / files[0].name)
events.head()

,session,aid,ts,type,locale
0,0,1517085,1659304800025,clicks,None
1,0,1563459,1659304904511,clicks,None
2,0,1309446,1659367439426,clicks,None
3,0,16246,1659367719997,clicks,None
4,0,1781822,1659367871344,clicks,None
